# This notebook converts the .mat data to netcdf format, 
# add the necessary metadata and plot the data

## load library

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os, sys
import json
import scipy.io
import cftime

In [2]:
# from src.netcdf import mat_to_xarray
sys.path.append('/Users/yugao/UOP/ORS-processing/src')
from metadata import create_json, process_attributes_direct
from netcdf_sbe37 import read_mat_file, mat_to_xarray, save_to_netcdf, create_xarray_dataset, extract_metadata

## INPUT REQUIRED: 
## Depth parameters from mooring diagram and mooring log 

In [3]:
instrument_height_above_bottom =  39   
water_depth_from_mooring_diagram = 4540      # water depth from mooring diagram
water_depth_from_ship_uncorrected = 4600     # uncorrected water depth 
water_depth_from_ship_corrected = 4565       # Replace with actual value, best water depth

#  Anchor (1) + chain (5) + nystron (20) + chain (5) + releases (1) + chain (5) 
# + 6 terminations at 0.25 ea (1.5) 
# + distance from termination on SBE cage to sensor (0.5) = 39 m

## read mat file

In [4]:
# mat_file_path = '../../data/external/stratus15/stratus15_sbe37_12257.mat'
mat_file_path = '../../data/external/stratus15/stratus15_sbe37_11394.mat'
mat_data = read_mat_file(mat_file_path)
# Inspect the structure of mat_data
# print(mat_data.keys()) 
netcdf_path = '/Users/yugao/UOP/ORS-processing/data/processed/stratus/stratus15' + mat_file_path[-16:-4] + '.nc'
json_path = '/Users/yugao/UOP/ORS-processing/data/metadata/stratus_OS_NTAS_2016_D_TS.json'
netcdf_path

'/Users/yugao/UOP/ORS-processing/data/processed/stratus/stratus15_sbe37_11394.nc'

## convert mat file to netcdf

In [5]:
mat_data = scipy.io.loadmat(mat_file_path, struct_as_record=False, squeeze_me=True)

# Extract and display basic information safely
print(mat_data.keys())
for key in mat_data.keys():
    if not key.startswith('__'):
        data = mat_data[key]
        data_type = type(data)
        data_shape = data.shape if hasattr(data, 'shape') else 'Scalar'
        print(f"{key}: {data_type}, shape: {data_shape}")

# Initialize an empty xarray dataset
ds = xr.Dataset()

# Handling time-series data with 'mday' as the time dimension
if 'mday' in mat_data:
    time_dim = 'time'
    ds[time_dim] = xr.DataArray(data=np.squeeze(mat_data['mday']), dims=[time_dim], attrs={'units': 'days since 0000-01-01 00:00:00', 'calendar': 'gregorian'})

    # Define metadata for variables
    variables = ['abssal', 'cond', 'sal', 'temp', 'press']
    units = {'temp': '°C', 'sal': 'psu', 'abssal': 'g/kg', 'cond': 'S/m', 'press': 'dbar'}
    long_names = {
        'temp': 'sea water temperature',
        'sal': 'practical salinity',
        'abssal': 'absolute salinity',
        'cond': 'sea water conductivity',
        'press': 'sea water pressure'
    }

    # Add other variables if they exist
    for var_name in variables:
        if var_name in mat_data:
            ds[var_name] = xr.DataArray(
                data=np.squeeze(mat_data[var_name]),
                dims=[time_dim],
                attrs={'units': units[var_name], 'long_name': long_names[var_name]}
            )
ds

dict_keys(['__header__', '__version__', '__globals__', 'abssal', 'cond', 'depth', 'latitude', 'longitude', 'mday', 'meta', 'press', 'sal', 'temp', 'yday', 'year'])
abssal: <class 'numpy.ndarray'>, shape: (96845,)
cond: <class 'numpy.ndarray'>, shape: (96845,)
depth: <class 'int'>, shape: Scalar
latitude: <class 'float'>, shape: Scalar
longitude: <class 'float'>, shape: Scalar
mday: <class 'numpy.ndarray'>, shape: (96845,)
meta: <class 'scipy.io.matlab._mio5_params.mat_struct'>, shape: Scalar
press: <class 'numpy.ndarray'>, shape: (96845,)
sal: <class 'numpy.ndarray'>, shape: (96845,)
temp: <class 'numpy.ndarray'>, shape: (96845,)
yday: <class 'numpy.ndarray'>, shape: (96845,)
year: <class 'int'>, shape: Scalar


<xarray.Dataset> Size: 5MB
Dimensions:  (time: 96845)
Coordinates:
  * time     (time) float64 775kB 7.365e+05 7.365e+05 ... 7.368e+05 7.368e+05
Data variables:
    abssal   (time) float64 775kB 0.0 28.08 25.72 25.54 ... nan nan nan nan
    cond     (time) float64 775kB 8e-05 2.978 2.752 2.724 ... -99.0 -99.0 -99.0
    sal      (time) float64 775kB 0.0 27.94 25.59 25.41 ... nan nan nan nan
    temp     (time) float64 775kB 18.01 6.287 6.293 6.136 ... 99.0 -48.6 99.0
    press    (time) float64 775kB 0.495 0.799 -0.192 ... -1.367e+04 -1.341e+04

## Meta data

In [6]:
# Investigate the contents of the 'meta' field to see what additional metadata it contains
meta_data = mat_data['meta']

# Import mat_struct from scipy.io.matlab
from scipy.io.matlab import mat_struct

# Initialize an empty dictionary to store the extracted useful information
useful_info = {}

# Iterate over the fields of the MATLAB structure object
for field_name in meta_data._fieldnames:
    # Skip the 'data' field
    if field_name == 'data':
        continue
    # Extract the value of the current field
    field_value = getattr(meta_data, field_name)
    # If the field value is another mat_struct, extract useful information recursively
    if isinstance(field_value, mat_struct):
        nested_info = {}
        # Iterate over the fields of the nested mat_struct
        for nested_field_name in field_value._fieldnames:
            nested_field_value = getattr(field_value, nested_field_name)
            nested_info[nested_field_name] = nested_field_value
        useful_info[field_name] = nested_info
    # If the field value is not a mat_struct, simply store it in the dictionary
    else:
        useful_info[field_name] = field_value

# add depth parameter
depth_parameters = {
    # consult mooring diagram
    'water_depth_from_mooring_diagram': water_depth_from_mooring_diagram,  
     # uncorrected water depth
    'water_depth_from_ship_uncorrected': water_depth_from_ship_uncorrected,
    # Replace with actual value, best water depth
    'water_depth_from_ship_corrected': water_depth_from_ship_corrected,        
    # instrument_depth_from_mooring_diagram = diagram depth  - height above bottom 
    'instrument_depth_from_mooring_diagram': water_depth_from_mooring_diagram - instrument_height_above_bottom,  
    # instrument_depth_from_mooring_log = corrected depth (4538.97 m) - height above bottom (39 m) 
    'instrument_depth_from_mooring_log': water_depth_from_ship_corrected - instrument_height_above_bottom,  
    'instrument_height_above_bottom': 39
}
# Initialize an empty dictionary to store the flattened attributes
attributes_flat = {}

# Function to flatten nested dictionaries
def flatten_dict(d, parent_key='', sep='_'):
    items = []
    for k, v in d.items():
        new_key = parent_key + sep + k if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        elif isinstance(v, np.ndarray):
            # Convert numpy array to list before adding to attributes
            items.append((new_key, v.tolist()))
        elif isinstance(v, scipy.io.matlab._mio5_params.mat_struct):
            # Convert mat_struct object to string representation
            continue
        else:
            items.append((new_key, v))
    return dict(items)


# Flatten the useful_info dictionary further
flattened_info = flatten_dict(useful_info)

# Flatten the 'global' dictionary further
flattened_global = flatten_dict(useful_info['global'], parent_key='global')

# Flatten the 'instrument' dictionary further
flattened_instrument = flatten_dict(useful_info['instrument'], parent_key='instrument')

# Flatten the 'history' dictionary further, excluding 'history_decode'
flattened_history = flatten_dict({k: v for k, v in useful_info['history'].items() if k != 'decode'}, parent_key='history')

# Update the attributes with the flattened dictionaries
attributes_flat.update(flattened_info)
# attributes_flat.update(flattened_platform)
attributes_flat.update(flattened_global)
attributes_flat.update(flattened_instrument)
attributes_flat.update(flattened_history)

# Update the attributes with the depth parameters
for key, value in depth_parameters.items():
    attributes_flat[f'depth_{key}'] = value

# Further process time and location attributes to make the data searchable
from metadata_sbe37 import process_attributes_direct

# Process the MATLAB data using the adjusted function
processed_attributes = process_attributes_direct(mat_data)

ds.attrs.update(processed_attributes)

# Parse the start date strings into cftime objects, assuming Gregorian calendar
time_coverage_start_str = ds.attrs['time_coverage_start']
time_ds = cftime.num2date(ds['time'][:] - ds['time'][0], units=f"days since {time_coverage_start_str}", calendar='gregorian')
time_ds
ds['time'] = time_ds
# Update the attributes of the xarray Dataset with the flattened dictionary
ds.attrs.update(attributes_flat)

## Save the dataset to netCDF

In [7]:
ds.to_netcdf(netcdf_path)

### Meta data: add depth parameters

In [8]:
ds.attrs

{'time_coverage_start': '2016-06-12T18:25:01Z',
 'time_coverage_end': '2017-05-15T00:45:01Z',
 'time_coverage_duration': 'P336D',
 'latitude_anchor_survey': -19.626223333333332,
 'longitude_anchor_survey': -84.94696666666667,
 'geospatial_lat_min': -19.626223333333332,
 'geospatial_lat_max': -19.626223333333332,
 'geospatial_lon_min': -84.94696666666667,
 'geospatial_lon_max': -84.94696666666667,
 'geospatial_vertical_min': 4528,
 'geospatial_vertical_max': 4528,
 'time_coverage_resolution': 'PT1H',
 'year': 2016,
 'site': 'Stratus',
 'deployment': '15',
 'principal_investigator': 'Robert Weller',
 'platform_experiment': 'STRATUS',
 'platform_type': 'surface mooring',
 'platform_deck_height_cm': 55,
 'platform_watch_circle_nm': 3.5,
 'platform_water_depth': 4565,
 'platform_wmo_platform_code': [],
 'platform_ndbc_platform_code': [],
 'platform_naming_authority': 'OceanSITES',
 'platform_anchor_times': [736501.9083333333, 736827.4694444444],
 'platform_anchor_over_time': '20-Jun-2016 21

### Variables

In [9]:
ds.temp

<xarray.DataArray 'temp' (time: 96845)> Size: 775kB
array([ 18.0059,   6.2874,   6.2925, ...,  99.    , -48.605 ,  99.    ])
Coordinates:
  * time     (time) object 775kB 2016-06-12 18:25:01 ... 2017-05-15 00:45:01....
Attributes:
    units:      °C
    long_name:  sea water temperature